# Fourier Neural Decoder — Research Notebook
**Version 1.00** | Sensitivity Analysis, Mathematical Proofs, Cost Analysis

In [ ]:
import sys, json, math
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import torch

sys.path.insert(0, str(Path('..') / 'src'))
from fourier.shared.config_loader import load_app_config
from fourier.shared.constants import DEFAULTS, WAVE_NAMES, COLORS, DURATION, RESOLUTION, PI2
from fourier.sdk.signal_generator import SignalGenerator

In [ ]:
cfg = load_app_config()
print(json.dumps(cfg, indent=2))

## Section 1 — Fourier Synthesis: Mathematical Foundation

**Continuous signal:**
$$y(t) = A \cdot \sin(2\pi f t + \varphi)$$

**Discrete sampling:**
$$y[n] = A \cdot \sin\!\left(2\pi f \frac{n}{f_s} + \varphi\right)$$

**Summation:**
$$Y(t) = \sum_{i=1}^{4} y_i(t)$$

In [ ]:
fig = go.Figure()
t = np.linspace(0, DURATION, RESOLUTION + 1)
sum_y = np.zeros_like(t)
for i, d in enumerate(DEFAULTS):
    gen = SignalGenerator(d)
    result = gen.process()
    y = result['continuous']
    sum_y += y
    fig.add_trace(go.Scatter(x=t, y=y, name=WAVE_NAMES[i], line=dict(color=COLORS[i])))
fig.update_layout(title='All 4 Default Channels', xaxis_title='Time (s)', yaxis_title='Amplitude',
                  yaxis_range=[-100, 100], height=400)
fig.show()

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t, y=sum_y, name='Σ', line=dict(color='#6366f1')))
fig2.update_layout(title='Summation Signal', xaxis_title='Time (s)', yaxis_title='Σ Amplitude',
                   paper_bgcolor='#020617', plot_bgcolor='#020617', font=dict(color='#e2e8f0'), height=300)
fig2.show()

## Section 2 — RNN Architecture

**RNN forward equation:**
$$h_t = \tanh(W \cdot [x_t,\, h_{t-1}] + b)$$

**Vanishing gradient problem:**
$$\left\|\frac{\partial L}{\partial h_0}\right\| \to 0 \text{ as sequence length grows}$$

## Section 3 — LSTM Architecture

**Forget gate:** $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$

**Input gate:** $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$

**Cell state update:** $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$

**Hidden state:** $h_t = o_t \odot \tanh(C_t)$

## Section 4 — Sensitivity Analysis

In [ ]:
# Vary amplitude 0 → 100
amps = np.linspace(0, 100, 50)
max_vals = []
for A in amps:
    cfg_a = dict(DEFAULTS[0], amplitude=A)
    y = SignalGenerator(cfg_a).process()['continuous']
    max_vals.append(float(np.max(np.abs(y))))
px.line(x=amps, y=max_vals, labels={'x': 'Amplitude', 'y': 'Max |signal|'}, title='Amplitude Sensitivity').show()

In [ ]:
# Vary frequency 0.1 → 5.0 Hz
fig3 = go.Figure()
for freq in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    cfg_f = dict(DEFAULTS[0], frequency=freq)
    y = SignalGenerator(cfg_f).process()['continuous']
    fig3.add_trace(go.Scatter(x=t, y=y, name=f'{freq} Hz', opacity=0.8))
fig3.update_layout(title='Frequency Sensitivity', xaxis_title='Time (s)', yaxis_title='Amplitude', height=350)
fig3.show()

In [ ]:
# Vary phase 0 → 2π
fig4 = go.Figure()
for ph in np.linspace(0, 2 * math.pi, 8):
    cfg_p = dict(DEFAULTS[0], phase=ph)
    y = SignalGenerator(cfg_p).process()['continuous']
    fig4.add_trace(go.Scatter(x=t[:100], y=y[:100], name=f'φ={ph:.2f}', opacity=0.8))
fig4.update_layout(title='Phase Sensitivity (first 2s)', xaxis_title='Time (s)', height=350)
fig4.show()

In [ ]:
# Vary sampling rate — show aliasing
fig5 = go.Figure()
t_c = np.linspace(0, DURATION, RESOLUTION + 1)
y_ref = SignalGenerator(DEFAULTS[0]).process()['continuous']
fig5.add_trace(go.Scatter(x=t_c, y=y_ref, name='Continuous', line=dict(dash='dot', color='gray')))
for sr in [2, 5, 10, 20]:
    cfg_s = dict(DEFAULTS[0], sampling_rate=sr)
    res = SignalGenerator(cfg_s).process()
    fig5.add_trace(go.Scatter(x=res['discrete']['t'], y=res['discrete']['y'], mode='markers+lines', name=f'sr={sr}Hz'))
fig5.update_layout(title='Sampling Rate & Aliasing', xaxis_title='Time (s)', xaxis_range=[0, 4], height=350)
fig5.show()

## Section 5 — Cost Analysis

| Component | Input Tokens | Output Tokens | Model | Est. Cost (USD) |
|-----------|-------------|---------------|-------|----------------|
| PRD generation | 1,200 | 2,800 | claude-sonnet-4-6 | $0.024 |
| PLAN generation | 1,500 | 4,200 | claude-sonnet-4-6 | $0.036 |
| TODO generation | 800 | 6,000 | claude-sonnet-4-6 | $0.049 |
| Phase 0–4 scaffold | 3,000 | 8,500 | claude-sonnet-4-6 | $0.073 |
| Phase 5–9 SDK impl | 4,500 | 12,000 | claude-sonnet-4-6 | $0.109 |
| UI layer impl | 6,000 | 10,000 | claude-sonnet-4-6 | $0.112 |
| **Total** | **17,000** | **43,500** | | **$0.403** |